# M3 實驗:Roboflow→EPFL(分階段) vs EPFL-only

**受控實驗**,回答:「先用 Roboflow 再用 EPFL,會不會比只用 EPFL 好?」

| | Arm A(基準) | Arm B(處理) |
|---|---|---|
| Stage 1 | 無 | COCO→Roboflow(11類)40 epoch |
| Stage 2 | COCO→EPFL 540 | stage1→EPFL 540 |
| Stage 2 設定 | **兩臂完全相同**:epochs=60, res=704, batch=4 | |

唯一差異 = 有沒有 Roboflow 前置階段。兩臂在**同一 EPFL test 集**上比 per-class P/R/F1,重點看 **刀具/容器**(Roboflow 唯一能影響的類)。

**用法**:GPU(T4)→ 全部執行 → 上傳 `data.zip`(**一個檔**,內含 EPFL 540 + Roboflow)。約 2~3.5 小時(4 段訓練)。
⚠ 含 EPFL(CC-NC)→ 驗證用、不出貨。

In [ ]:
# 1) 安裝
!nvidia-smi -L
!pip -q install "rfdetr[train,loggers]" supervision

In [ ]:
# 2) 上傳 data.zip(一個檔,內含 epfl/ 與 rf/)並解壓
from google.colab import files
print('請上傳 data.zip(內含 EPFL 540 與 Roboflow)')
up = files.upload()
!unzip -q -o data.zip -d /content
!echo EPFL: && ls /content/epfl && echo RF: && ls /content/rf

In [ ]:
# 3) 評估工具(同 m3_finetune 的 eval_model;test = EPFL test)
import os, json, gc
from collections import defaultdict
from PIL import Image
try:
    import torch
except Exception:
    torch = None

TEST = '/content/epfl/test'
NAMES = {0:'人',1:'刀具',2:'砧板',3:'食材',4:'鍋鏟',5:'鍋子',6:'手',7:'容器',8:'抹布',9:'夾子',10:'手套'}
CAT2CLS = {i: i-1 for i in range(1, 12)}

def free():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def _load_gt(test_dir):
    j = json.load(open(os.path.join(test_dir, '_annotations.coco.json'), encoding='utf-8'))
    id2fn = {im['id']: im['file_name'] for im in j['images']}
    by = {}
    for a in j['annotations']:
        x, y, w, h = a['bbox']
        by.setdefault(id2fn[a['image_id']], []).append((a['category_id'], [x, y, x+w, y+h]))
    return by

def eval_model(model, test_dir, cat_to_class=None, thr=0.3, iou_thr=0.5):
    gt = _load_gt(test_dir)
    TP = defaultdict(int); FP = defaultdict(int); FN = defaultdict(int); NG = defaultdict(int)
    for fn, objs in gt.items():
        det = model.predict(Image.open(os.path.join(test_dir, fn)).convert('RGB'), threshold=thr)
        pboxes = det.xyxy.tolist() if len(det) else []
        pcls = [int(c) for c in det.class_id] if len(det) else []
        if len(det) and getattr(det, 'confidence', None) is not None:
            pconf = [float(x) for x in det.confidence]
        else:
            pconf = [1.0] * len(pboxes)
        gts = [(cat_to_class.get(gc, -1), gb) for gc, gb in objs]
        for gc, _ in gts:
            NG[gc] += 1
        mg = [False] * len(gts)
        for i in sorted(range(len(pboxes)), key=lambda k: -pconf[k]):
            c, box = pcls[i], pboxes[i]
            best, bestv = -1, iou_thr
            for j, (gc, gb) in enumerate(gts):
                if mg[j] or gc != c:
                    continue
                v = _iou(box, gb)
                if v >= bestv:
                    bestv, best = v, j
            if best >= 0:
                mg[best] = True; TP[c] += 1
            else:
                FP[c] += 1
        for j, (gc, gb) in enumerate(gts):
            if not mg[j]:
                FN[gc] += 1
    out = {'per_class': {}}
    tTP = tFP = tFN = 0
    for gc in sorted(NG):
        tp, fp, fn = TP[gc], FP[gc], FN[gc]
        tTP += tp; tFP += fp; tFN += fn
        p = tp/(tp+fp) if (tp+fp) else 0.0
        r = tp/(tp+fn) if (tp+fn) else 0.0
        f = 2*p*r/(p+r) if (p+r) else 0.0
        out['per_class'][NAMES.get(gc, gc)] = {'n_gt': NG[gc], 'precision': round(p,3), 'recall': round(r,3), 'f1': round(f,3)}
    P = tTP/(tTP+tFP) if (tTP+tFP) else 0.0
    R = tTP/(tTP+tFN) if (tTP+tFN) else 0.0
    F = 2*P*R/(P+R) if (P+R) else 0.0
    out['overall'] = {'precision': round(P,3), 'recall': round(R,3), 'f1': round(F,3)}
    return out

print('工具就緒。EPFL test 圖數:', len(_load_gt(TEST)))

In [ ]:
# 4) Arm A:EPFL-only(基準)。COCO→EPFL 540
from rfdetr import RFDETRNano
print('===== Arm A:EPFL-only =====')
mA = RFDETRNano()
mA.train(dataset_dir='/content/epfl', epochs=60, batch_size=4, grad_accum_steps=4,
         lr=1e-4, resolution=704, output_dir='/content/out_A')
mA = None; free()
ftA = RFDETRNano(pretrain_weights='/content/out_A/checkpoint_best_regular.pth', num_classes=11)
resA = eval_model(ftA, TEST, cat_to_class=CAT2CLS)
print('Arm A overall:', resA['overall'])
ftA = None; free()

In [ ]:
# 5) Arm B:Roboflow → EPFL(分階段)。stage2 設定與 Arm A 完全相同
from rfdetr import RFDETRNano
print('===== Arm B:stage1 Roboflow =====')
s1 = RFDETRNano()
s1.train(dataset_dir='/content/rf', epochs=40, batch_size=4, grad_accum_steps=4,
         lr=1e-4, resolution=704, output_dir='/content/out_B_stage1')
s1 = None; free()
print('===== Arm B:stage2 EPFL(從 stage1 權重) =====')
s2 = RFDETRNano(pretrain_weights='/content/out_B_stage1/checkpoint_best_regular.pth', num_classes=11)
s2.train(dataset_dir='/content/epfl', epochs=60, batch_size=4, grad_accum_steps=4,
         lr=1e-4, resolution=704, output_dir='/content/out_B_final')
s2 = None; free()
ftB = RFDETRNano(pretrain_weights='/content/out_B_final/checkpoint_best_regular.pth', num_classes=11)
resB = eval_model(ftB, TEST, cat_to_class=CAT2CLS)
print('Arm B overall:', resB['overall'])
ftB = None; free()

In [ ]:
# 6) 比較 + 存檔下載
print('======== 實驗結果:EPFL-only(A) vs Roboflow->EPFL(B) ========')
print()
print('{:<8}{:>10}{:>10}{:>10}'.format('', 'Prec', 'Recall', 'F1'))
for name, res in [('Arm A', resA), ('Arm B', resB)]:
    o = res['overall']
    print('{:<8}{:>10}{:>10}{:>10}'.format(name, o['precision'], o['recall'], o['f1']))

print()
print('重點類別(Roboflow 唯一能影響):刀具 / 容器')
print('{:<6}{:>8}{:>8}{:>8}   {:>8}{:>8}{:>8}'.format('類別', 'A_P', 'A_R', 'A_F1', 'B_P', 'B_R', 'B_F1'))
for cls in ['刀具', '容器']:
    a = resA['per_class'].get(cls, {})
    b = resB['per_class'].get(cls, {})
    print('{:<6}{:>8}{:>8}{:>8}   {:>8}{:>8}{:>8}'.format(
        cls, a.get('precision'), a.get('recall'), a.get('f1'),
        b.get('precision'), b.get('recall'), b.get('f1')))

out = {'arm_A_epfl_only': resA, 'arm_B_roboflow_then_epfl': resB}
json.dump(out, open('/content/experiment_results.json', 'w'), ensure_ascii=False, indent=2)
print()
print('已存 /content/experiment_results.json')
from google.colab import files
files.download('/content/experiment_results.json')
for p in ['/content/out_A/checkpoint_best_regular.pth', '/content/out_B_final/checkpoint_best_regular.pth']:
    try:
        files.download(p)
    except Exception as e:
        print('下載失敗', p, e)

## 怎麼判讀

- **看刀具 / 容器**(Roboflow 唯一能影響的類):
  - Arm B 明顯 > Arm A → **先 Roboflow 有幫助**。
  - 相近或更差 → **沒幫助**(符合預期:換 11 類時偵測頭重置 + COCO 已涵蓋刀/容器)。
- 其他類兩臂應相近(當作「無差異對照」)。
- ⚠ test 僅 90 圖,單類差幾個框就會晃 → **差距要夠大**才算數。

把上面的比較表 + `experiment_results.json` 貼回給 Claude 一起判讀。